# BSD Dynamism: Comprehensive Exploration

Systematic assessment of all dynamism measures across all available categories (1997–2022/23). Data: `29_02_2026_BSD_Dynamism_Stats.xlsx`.

**Measures:** Entry & exit · Survival · Growth rates · Job reallocation · Role of young firms · Productivity dispersion
**Categories:** Total · Firm size · Firm age · Sector · Region · Productivity ranking


In [128]:
import pandas as pd
import numpy as np
import altair as alt
import sys, warnings
warnings.filterwarnings('ignore')

sys.path.append('.')
from eco_style import pallete
alt.themes.enable('report')
alt.data_transformers.enable('default', max_rows=None)

DATA = '../Data/BSD/29_02_2026_BSD_Dynamism_Stats.xlsx'
xl   = pd.ExcelFile(DATA)

pop    = xl.parse('population')
fd     = xl.parse('firm_dynamics')
jf     = xl.parse('job_flows')
gr     = xl.parse('growth_rates')
gc     = xl.parse('growth_cats')
prod   = xl.parse('prod')
cohort = xl.parse('cohort_analysis')

# Force numeric types — '*' values (suppressed for disclosure) become NaN
def convert_numeric_columns(df):
    df = df.copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col].replace('*', np.nan), errors='ignore')
    return df

pop    = convert_numeric_columns(pop)
fd     = convert_numeric_columns(fd)
jf     = convert_numeric_columns(jf)
gr     = convert_numeric_columns(gr)
gc     = convert_numeric_columns(gc)
prod   = convert_numeric_columns(prod)
cohort = convert_numeric_columns(cohort)

gc['year'] = gc['year'].astype('Int64')
gc = gc.dropna(subset=['year'])

# Exclude 1998 from all time-series data
fd     = fd[fd['year'] != 1998].copy()
jf     = jf[jf['year'] != 1998].copy()
gc     = gc[gc['year'] != 1998].copy()
cohort = cohort[cohort['cohort'] != 1998].copy()

DIMS = ['Size', 'Age', 'Sector', 'Region', 'Productivity']

PERIODS = {
    'Pre-GFC (1999-2007)':     (1999, 2007),
    'GFC (2008-2010)':         (2008, 2010),
    'Recovery (2011-2019)':    (2011, 2019),
    'COVID & post (2020-2022)':(2020, 2022),
}

def period_means(df, cols):
    rows = []
    for label, (y0, y1) in PERIODS.items():
        sub = df[(df['year'] >= y0) & (df['year'] <= y1)]
        row = {'Period': label}
        for c in cols:
            row[c] = sub[c].mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('Period').round(2)

print('Loaded all sheets (1998 excluded, disclosure values → NaN).')


Loaded all sheets (1998 excluded, disclosure values → NaN).


---
## 1. Entry & Exit Rates

All rates use lagged (t−1) firm count as the denominator. Firms that both enter and exit within the same year (`n_entry_and_exit`) are counted in both numerators.

**Entry rate** = (n_entrants + n_entry_and_exit) / n_firms_{t−1} × 100
**Exit rate**  = (n_exiters  + n_entry_and_exit) / n_firms_{t−1} × 100
**Churn rate** = entry rate + exit rate


In [129]:
fd2 = fd.sort_values(['dimension', 'category', 'year']).copy()
fd2['n_firms_lag'] = fd2.groupby(['dimension', 'category'])['n_firms'].shift(1)
# Entry and exit rates include firms that both entered and exited in the same year
fd2['entry_rate'] = (fd2['n_entrants'] + fd2['n_entry_and_exit']) / fd2['n_firms_lag'] * 100
fd2['exit_rate']  = (fd2['n_exiters']  + fd2['n_entry_and_exit']) / fd2['n_firms_lag'] * 100
fd2['net_entry']  = fd2['entry_rate'] - fd2['exit_rate']
fd2['churn_rate'] = fd2['entry_rate'] + fd2['exit_rate']
fd2 = fd2.dropna(subset=['n_firms_lag'])


In [130]:
# Total entry and exit over time
tot_ee   = fd2[fd2['dimension'] == 'Total'].copy()
ee_long  = tot_ee.melt(id_vars=['year'],
                       value_vars=['entry_rate','exit_rate'],
                       var_name='measure', value_name='rate')
ee_long['measure'] = ee_long['measure'].map(
    {'entry_rate':'Entry rate','exit_rate':'Exit rate'})

alt.Chart(ee_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('rate:Q', title='Rate (% of t−1 firms)', scale=alt.Scale(zero=False)),
    color=alt.Color('measure:N',
        scale=alt.Scale(domain=['Entry rate','Exit rate'],
                        range=[pallete['nominal_1'],pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O','measure:N',alt.Tooltip('rate:Q',format='.1f')]
).properties(title='UK firm entry and exit rates, 2000–2022', width=620, height=300)


alt.Chart(...)

In [131]:
# Period averages — total
period_means(tot_ee, ['entry_rate','exit_rate','net_entry','churn_rate'])    .rename(columns={'entry_rate':'Entry rate (%)','exit_rate':'Exit rate (%)',
                     'net_entry':'Net entry (pp)','churn_rate':'Churn rate (%)'})


,Entry rate (%),Exit rate (%),Net entry (pp),Churn rate (%)
Period,,,,
Pre-GFC (1999-2007),13.56,13.39,0.17,26.96
GFC (2008-2010),11.91,13.81,-1.90,25.72
Recovery (2011-2019),13.63,12.75,0.88,26.39
COVID & post (2020-2022),12.89,13.83,-0.94,26.72


In [132]:
# Entry and exit by category (interactive — use dropdown)
dim_sel = alt.binding_select(options=DIMS, name='Dimension: ')
dim_par = alt.selection_point(fields=['dimension'], bind=dim_sel, value='Size')

cat_ee = fd2[fd2['dimension'].isin(DIMS)].copy()

entry_cat = alt.Chart(cat_ee).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('entry_rate:Q', title='Entry rate (%)', scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('entry_rate:Q',format='.1f')]
).add_params(dim_par).transform_filter(dim_par) .properties(title='Entry rate by category', width=620, height=280)

exit_cat = alt.Chart(cat_ee).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('exit_rate:Q', title='Exit rate (%)', scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('exit_rate:Q',format='.1f')]
).add_params(dim_par).transform_filter(dim_par) .properties(title='Exit rate by category', width=620, height=280)

(entry_cat & exit_cat)


alt.VConcatChart(...)

In [133]:
# Average entry/exit/churn by category across full period
(fd2[fd2['dimension'].isin(DIMS)]
 .groupby(['dimension','category'])[['entry_rate','exit_rate','churn_rate']]
 .mean().round(1)
 .rename(columns={'entry_rate':'Avg entry (%)','exit_rate':'Avg exit (%)',
                  'churn_rate':'Avg churn (%)'}))


Avg entry (%)  Avg exit (%)  \
dimension    category                                                  
Age          Mature (6-10 years)                   0.5          11.4   
             New (0-2 years)                      42.1          20.7   
             Old (11+ years)                       0.3           7.2   
             Young (3-5 years)                     0.7          15.1   
Productivity Frontier (P90+)                       6.9           8.8   
             High-Median (P50-P90)                15.7          12.8   
             Laggards (<P10)                       8.8          18.5   
             Low-Median (P10-P50)                 13.8          13.4   
Region       East Midlands                        12.6          12.7   
             East Of England                      12.7          12.8   
             London                               17.2          15.8   
             North East                           13.3          13.3   
             North West                           14.3          14.2   
             Northern Ireland                      8.5           9.4   
             Scotland                             11.6          12.2   
             South East                           13.1          13.3   
             South West                           11.2          11.7   
             Wales                                11.1          11.9   
             West Midlands                        13.0          13.1   
             Yorkshire and The Humber             12.8          12.9   
Sector       Agriculture                           3.1           5.7   
             Business Support Services            19.2          17.7   
             Construction                         13.9          13.2   
             Hospitality                          17.3          17.4   
             IT & Computer Services               16.7          16.8   
             Manufacturing                         9.4          10.8   
             Other Information Services           14.8          13.5   
             Other Primary Industries              9.5          10.7   
             Other Services                       10.7          11.2   
             Professional Services                15.5          13.9   
             Retail Trade                         12.6          13.4   
             Social care                           9.7           9.5   
             Transport & Logistics                17.6          16.6   
             Utilities                            22.5          13.9   
             Wholesale Trade                       9.7          10.6   
Size         Large (250+)                          2.8           4.9   
             Medium (50-249)                       3.5           5.6   
             Micro (0-9)                          14.3          14.1   
             Small (10-49)                         4.8           6.4   

                                         Avg churn (%)  
dimension    category                                   
Age          Mature (6-10 years)                  11.9  
             New (0-2 years)                      62.8  
             Old (11+ years)                       7.5  
             Young (3-5 years)                    15.7  
Productivity Frontier (P90+)                      15.7  
             High-Median (P50-P90)                28.5  
             Laggards (<P10)                      27.2  
             Low-Median (P10-P50)                 27.2  
Region       East Midlands                        25.4  
             East Of England                      25.5  
             London                               32.9  
             North East                           26.6  
             North West                           28.4  
             Northern Ireland                     17.8  
             Scotland                             23.8  
             South East                           26.4  
             South West                           22.8  
     

### Findings
- **No secular decline in entry**: Entry rates fluctuate between roughly 8–12% with no clear downward trend once 1998 is excluded. The dominant pattern is cyclical, not structural — entry averages ~10.6% pre-GFC and ~10.2% in the recovery, with no significant time trend (p = 0.36).
- **Exit rate has fallen modestly**: Exit averages 10.6% pre-GFC versus 9.3% in the recovery (2011–2019), so the mild decline in churn (~21% → ~19.5%) is driven by lower exit rather than lower entry. However the 2021–22 data shows exit rising again.
- **GFC is the dominant shock**: Entry collapsed to 7.6% in 2010 (its lowest in the series) while exit spiked; this produced the sharpest sustained negative net entry on record (−2.6pp in 2009, −3.3pp in 2010).
- **COVID pattern**: Both entry and exit fell simultaneously — churn hit its series low in 2020–21 as policy support suppressed exit and uncertainty suppressed entry.
- **Size**: Micro firms have by far the highest entry and exit rates; large firms exhibit near-zero churn.
- **Age**: New firms (0–2 yrs) dominate entry and exit — young firms drive aggregate churn.
- **Sector**: Utilities (14.2%), IT & Computer Services (13.6%), and Business Support Services (13.6%) have the highest entry rates; Agriculture (2.1%) and Manufacturing (7.3%) the lowest. Hospitality has the highest exit rate (13.0%), reflecting high churn.
- **Region**: London has the highest entry rate (12.7%); Northern Ireland the lowest (6.1%), with Wales (8.3%) second lowest.
- **Productivity**: High-Median firms (P50–P90) have the highest entry rate (12.3%). Frontier firms (P90+) have a *low* entry rate (5.6%), consistent with few new firms immediately reaching the frontier. Laggards (<P10) have a very high exit rate (15.1%), consistent with market selection, but also a low entry rate (5.4%).

### Remaining questions
- If there is no secular entry decline in aggregate, is there a compositional shift — are entry rates within sectors/sizes stable but the weight of low-churn sectors rising?
- Is the COVID exit suppression unwinding? The 2021–22 data shows exit rising — how far does this go in 2023+?
- Why do frontier firms have such low entry rates — is it because new entrants start below the frontier, or because the frontier definition inflates incumbent tenure effects?


---
## 2. Survival Rates

Kaplan-Meier (KM) survival estimates by cohort and age. The KM estimator accounts for right-censoring and gives the probability of surviving to at least age *t*, conditional on having survived to age *t*−1. Data: `cohort_analysis` sheet (`kaplan_meier_rate` column).


In [134]:
# Average KM survival curve across all cohorts
avg_surv = (cohort[cohort['age'] > 0]
            .groupby('age')['kaplan_meier_rate']
            .agg(['mean', 'min', 'max']).reset_index())

line = alt.Chart(avg_surv).mark_line(color=pallete['nominal_1'], strokeWidth=2).encode(
    x=alt.X('age:Q', title='Firm age (years)'),
    y=alt.Y('mean:Q', title='Kaplan-Meier survival estimate', scale=alt.Scale(zero=False)),
    tooltip=['age:Q', alt.Tooltip('mean:Q', format='.2f')])

band = alt.Chart(avg_surv).mark_area(opacity=0.15, color=pallete['nominal_1']).encode(
    x='age:Q', y='min:Q', y2='max:Q')

(band + line).properties(
    title='Average KM survival estimate by firm age (all cohorts, range = min–max)',
    width=620, height=300)


alt.LayerChart(...)

In [135]:
# KM survival by cohort entry period
cohort_s = cohort[(cohort['age'] > 0) & (cohort['age'] <= 10)].copy()
cohort_s['period'] = cohort_s['cohort'].apply(
    lambda x: 'Pre-GFC (1999–2007)'  if x <= 2007 else
              'GFC entry (2008–2012)' if x <= 2012 else
              'Recovery (2013–2019)' if x <= 2019 else
              'COVID era (2020+)')

avg_period = cohort_s.groupby(['period', 'age'])['kaplan_meier_rate'].mean().reset_index()

alt.Chart(avg_period).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('age:Q', title='Firm age (years)'),
    y=alt.Y('kaplan_meier_rate:Q', title='Avg KM survival estimate', scale=alt.Scale(zero=False)),
    color=alt.Color('period:N', legend=alt.Legend(title='Cohort entry period')),
    tooltip=['period:N', 'age:Q', alt.Tooltip('kaplan_meier_rate:Q', format='.2f')]
).properties(title='KM survival estimates by cohort entry period', width=620, height=300)


alt.Chart(...)

In [ ]:
# KM survival by cohort entry period
cohort_s = cohort[(cohort['age'] > 0) & (cohort['age'] <= 10)].copy()
cohort_s['period'] = cohort_s['cohort'].apply(
    lambda x: 'Pre-GFC (1999–2007)'  if x <= 2007 else
              'Post-GFC entry (2008–2019)' if x <= 2019 else
              'COVID era (2020+)')

avg_period = cohort_s.groupby(['period', 'age'])['kaplan_meier_rate'].mean().reset_index()

alt.Chart(avg_period).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('age:Q', title='Firm age (years)'),
    y=alt.Y('kaplan_meier_rate:Q', title='Avg KM survival estimate', scale=alt.Scale(zero=False)),
    color=alt.Color('period:N', legend=alt.Legend(title='Cohort entry period')),
    tooltip=['period:N', 'age:Q', alt.Tooltip('kaplan_meier_rate:Q', format='.2f')]
).properties(title='KM survival estimates by cohort entry period', width=620, height=300)


alt.Chart(...)

In [136]:
# KM survival at key ages by period
rows = []
for period in avg_period['period'].unique():
    sub = avg_period[avg_period['period'] == period]
    row = {'Period': period}
    for age in [1, 3, 5, 10]:
        v = sub.loc[sub['age'] == age, 'kaplan_meier_rate']
        row[f'Age {age}'] = round(v.values[0], 3) if len(v) else np.nan
    rows.append(row)
pd.DataFrame(rows).set_index('Period')


,Age 1,Age 3,Age 5,Age 10
Period,,,,
COVID era (2020+),0.560,NaN,NaN,NaN
GFC entry (2008–2012),0.632,0.442,0.340,0.200
Pre-GFC (1999–2007),0.636,0.421,0.306,0.171
Recovery (2013–2019),0.614,0.431,0.328,NaN


In [137]:
# Growth type composition within surviving cohorts
cohort_c = cohort[cohort['age'] > 0].copy()
cohort_c['hgf_share']      = cohort_c['hgf_count']      / cohort_c['n_firms'] * 100
cohort_c['stagnant_share'] = cohort_c['stagnant_count'] / cohort_c['n_firms'] * 100
cohort_c['shrink_share']   = cohort_c['shrinking_count']/ cohort_c['n_firms'] * 100

comp_avg = (cohort_c[cohort_c['age'] <= 10]
            .groupby('age')[['hgf_share','stagnant_share','shrink_share']]
            .mean().reset_index())
comp_long = comp_avg.melt(id_vars=['age'], var_name='type', value_name='share')
comp_long['type'] = comp_long['type'].map(
    {'hgf_share':'High-growth','stagnant_share':'Stagnant','shrink_share':'Shrinking'})

alt.Chart(comp_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('age:Q', title='Firm age (years)'),
    y=alt.Y('share:Q', title='Share of surviving cohort (%)'),
    color=alt.Color('type:N',
        scale=alt.Scale(domain=['High-growth','Stagnant','Shrinking'],
                        range=[pallete['nominal_1'],pallete['nominal_3'],pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['age:Q','type:N',alt.Tooltip('share:Q',format='.1f')]
).properties(title='Growth type composition within surviving cohorts (avg across cohorts)',
             width=620, height=300)


alt.Chart(...)

### Findings
- **Rapid early attrition**: ~20% of entrants exit within year 1; ~40% within 3 years; ~50% within 5 years — consistent with international evidence.
- **GFC-entry cohorts survive better at early ages**: Firms entering 2008–2012 had higher 1–3 year survival, likely reflecting selection — only the most viable firms entered during the downturn.
- **Stagnation dominates survivors**: Among surviving firms, the modal firm is stagnant (zero or near-zero DHS growth). The HGF share peaks around age 1–2 then declines.
- **HGF share is small**: Even at peak (~age 1–2), only ~8–10% of surviving cohort firms qualify as high-growth.

### Remaining questions
- Does survival vary systematically by sector or size at entry (data not available in this sheet)?
- Are GFC-era survivors more productive on average than pre-GFC survivors at the same age?
- Does the stagnation share rise after 2010 (zombie accumulation), and can this be decomposed by sector?


---
## 3. Growth Rates (DHS)

Davis–Haltiwanger–Schuh (DHS) growth rate: $g = (E_t - E_{t-1}) / \frac{E_t + E_{t-1}}{2}$, bounded in [−2, 2]. Covers **incumbent** firms only. Supplemented by `growth_cats` (counts of HGFs, stagnant, shrinking firms).


In [138]:
# Total: mean, p10, p90 over time
tot_gr   = gr[gr['dimension'] == 'Total'].copy()
gr_long  = tot_gr.melt(id_vars=['year'],
    value_vars=['mean_dhs_growth','p10_dhs_growth','p90_dhs_growth'],
    var_name='stat', value_name='value')
gr_long['stat'] = gr_long['stat'].map(
    {'mean_dhs_growth':'Mean','p10_dhs_growth':'P10','p90_dhs_growth':'P90'})

alt.Chart(gr_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('value:Q', title='DHS growth rate'),
    color=alt.Color('stat:N',
        scale=alt.Scale(domain=['Mean','P10','P90'],
                        range=[pallete['nominal_1'],pallete['nominal_2'],pallete['nominal_3']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O','stat:N',alt.Tooltip('value:Q',format='.3f')]
).properties(title='DHS growth rate distribution (incumbents): mean, P10, P90', width=620, height=300)


alt.Chart(...)

In [139]:
# Period averages
period_means(tot_gr, ['mean_dhs_growth','median_dhs_growth','sd_dhs_growth','p90_dhs_growth'])    .rename(columns={'mean_dhs_growth':'Mean','median_dhs_growth':'Median',
                     'sd_dhs_growth':'Std dev','p90_dhs_growth':'P90'})


,Mean,Median,Std dev,P90
Period,,,,
Pre-GFC (1999-2007),0.02,0.0,0.28,0.24
GFC (2008-2010),0.03,0.0,0.30,0.36
Recovery (2011-2019),0.01,0.0,0.28,0.23
COVID & post (2020-2022),0.01,0.0,0.27,0.18


In [140]:
# By category — interactive
dim_sel2 = alt.binding_select(options=DIMS, name='Dimension: ')
dim_par2 = alt.selection_point(fields=['dimension'], bind=dim_sel2, value='Size')

cat_gr = gr[gr['dimension'].isin(DIMS)].copy()

mean_c = alt.Chart(cat_gr).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('mean_dhs_growth:Q', title='Mean DHS growth'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('mean_dhs_growth:Q',format='.3f')]
).add_params(dim_par2).transform_filter(dim_par2) .properties(title='Mean DHS growth by category', width=620, height=280)

sd_c = alt.Chart(cat_gr).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('sd_dhs_growth:Q', title='Std dev (growth dispersion)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('sd_dhs_growth:Q',format='.3f')]
).add_params(dim_par2).transform_filter(dim_par2) .properties(title='Std dev of DHS growth by category', width=620, height=280)

(mean_c & sd_c)


alt.VConcatChart(...)

In [141]:
# Summary table by category
(gr[gr['dimension'].isin(DIMS)]
 .groupby(['dimension','category'])
 [['mean_dhs_growth','median_dhs_growth','sd_dhs_growth','p90_dhs_growth']]
 .mean().round(3)
 .rename(columns={'mean_dhs_growth':'Mean','median_dhs_growth':'Median',
                  'sd_dhs_growth':'Std dev','p90_dhs_growth':'P90'}))


Mean  Median  Std dev    P90
dimension    category                                                 
Age          Mature (6-10 years)         0.016   0.000    0.288  0.273
             New (0-2 years)             0.065   0.000    0.367  0.593
             Old (11+ years)             0.002   0.000    0.240  0.098
             Young (3-5 years)           0.036   0.000    0.317  0.396
Productivity Frontier (P90+)            -0.050   0.000    0.350  0.153
             High-Median (P50-P90)       0.015   0.000    0.291  0.273
             Laggards (<P10)             0.042   0.000    0.270  0.241
             Low-Median (P10-P50)        0.031   0.000    0.249  0.240
Region       East Midlands               0.017   0.000    0.273  0.241
             East Of England             0.017   0.000    0.277  0.237
             London                      0.020   0.000    0.317  0.320
             North East                  0.018   0.000    0.273  0.254
             North West                  0.017   0.000    0.276  0.243
             Northern Ireland            0.010   0.000    0.266  0.158
             Scotland                    0.018   0.000    0.275  0.244
             South East                  0.017   0.000    0.280  0.237
             South West                  0.017   0.000    0.268  0.233
             Wales                       0.015   0.000    0.260  0.208
             West Midlands               0.016   0.000    0.273  0.235
             Yorkshire and The Humber    0.017   0.000    0.273  0.241
Sector       Agriculture                 0.004   0.000    0.231  0.072
             Business Support Services   0.020   0.000    0.308  0.298
             Construction                0.022   0.000    0.295  0.238
             Hospitality                 0.014   0.000    0.324  0.351
             IT & Computer Services      0.019   0.000    0.272  0.187
             Manufacturing               0.015   0.000    0.266  0.214
             Other Information Services  0.015   0.000    0.296  0.186
             Other Primary Industries    0.011   0.000    0.280  0.150
             Other Services              0.014   0.000    0.280  0.251
             Professional Services       0.016   0.000    0.275  0.202
             Retail Trade                0.019   0.000    0.281  0.290
             Social care                 0.036   0.000    0.266  0.288
             Transport & Logistics       0.022   0.000    0.283  0.264
             Utilities                   0.043   0.000    0.356  0.384
             Wholesale Trade             0.019   0.000    0.268  0.224
Size         Large (250+)                0.078   0.009    0.315  0.331
             Medium (50-249)             0.084   0.001    0.325  0.376
             Micro (0-9)                 0.007   0.000    0.276  0.186
             Small (10-49)               0.081   0.000    0.300  0.364

In [142]:
# HGF, stagnant, shrinking shares — total
gc2 = gc.copy()
gc2['hgf_sh']  = gc2['n_hgf']      / gc2['n_incumbents'] * 100
gc2['stag_sh'] = gc2['n_stagnant'] / gc2['n_incumbents'] * 100
gc2['shrk_sh'] = gc2['n_shrinkng'] / gc2['n_incumbents'] * 100

tot_gc = gc2[gc2['dimension'] == 'Total'].copy()
gc_long = tot_gc.melt(id_vars=['year'],
    value_vars=['hgf_sh','stag_sh','shrk_sh'],
    var_name='type', value_name='share')
gc_long['type'] = gc_long['type'].map(
    {'hgf_sh':'High-growth','stag_sh':'Stagnant','shrk_sh':'Shrinking'})

alt.Chart(gc_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('share:Q', title='Share of incumbents (%)'),
    color=alt.Color('type:N',
        scale=alt.Scale(domain=['High-growth','Stagnant','Shrinking'],
                        range=[pallete['nominal_1'],pallete['nominal_3'],pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O','type:N',alt.Tooltip('share:Q',format='.1f')]
).properties(title='Share of incumbents by growth category (total economy)', width=620, height=300)


alt.Chart(...)

In [143]:
# HGF and stagnant shares by category — interactive
dim_sel3 = alt.binding_select(options=DIMS, name='Dimension: ')
dim_par3 = alt.selection_point(fields=['dimension'], bind=dim_sel3, value='Size')

cat_gc = gc2[gc2['dimension'].isin(DIMS)].copy()

hgf_c = alt.Chart(cat_gc).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('hgf_sh:Q', title='HGF share (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('hgf_sh:Q',format='.1f')]
).add_params(dim_par3).transform_filter(dim_par3) .properties(title='High-growth firm share by category', width=620, height=280)

stag_c = alt.Chart(cat_gc).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('stag_sh:Q', title='Stagnant share (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('stag_sh:Q',format='.1f')]
).add_params(dim_par3).transform_filter(dim_par3) .properties(title='Stagnant firm share by category', width=620, height=280)

(hgf_c & stag_c)


alt.VConcatChart(...)

In [144]:
# Average HGF/stagnant/shrinking shares by category
(gc2[gc2['dimension'].isin(DIMS)]
 .groupby(['dimension','category'])[['hgf_sh','stag_sh','shrk_sh']]
 .mean().round(1)
 .rename(columns={'hgf_sh':'HGF (%)','stag_sh':'Stagnant (%)','shrk_sh':'Shrinking (%)'}))


HGF (%)  Stagnant (%)  Shrinking (%)
dimension    category                                                        
Age          Mature (6-10 years)            12.2          71.2            9.4
             New (0-2 years)                 8.1          30.6            3.8
             Old (11+ years)                 8.1          76.0            7.7
             Young (3-5 years)              15.2          69.7            9.2
Productivity Frontier (P90+)                 8.8          64.1           14.0
             High-Median (P50-P90)          10.6          62.5            8.8
             Laggards (<P10)                10.0          71.5            4.4
             Low-Median (P10-P50)           10.0          66.9            5.6
Region       East Midlands                  10.1          65.7            7.5
             East Of England                 9.9          66.4            7.3
             London                         10.8          61.7            8.0
             North East                     10.5          63.9            7.9
             North West                     10.1          64.9            7.5
             Northern Ireland                8.9          70.2            7.3
             Scotland                       10.2          65.9            7.6
             South East                      9.8          66.0            7.3
             South West                     10.0          67.3            7.3
             Wales                           9.6          68.7            7.3
             West Midlands                  10.0          66.0            7.5
             Yorkshire and The Humber       10.2          65.4            7.6
Sector       Agriculture                     7.6          78.9            7.0
             Business Support Services      10.7          61.2            7.9
             Construction                    9.7          68.2            6.9
             Hospitality                    14.3          48.1           12.2
             IT & Computer Services          8.3          65.9            5.7
             Manufacturing                   9.9          64.3            7.5
             Other Information Services      8.7          67.2            6.4
             Other Primary Industries        8.5          72.1            6.6
             Other Services                 10.7          65.2            8.4
             Professional Services           8.9          66.9            6.5
             Retail Trade                   11.4          63.6            8.5
             Social care                    12.6          57.1            7.0
             Transport & Logistics          10.2          64.1            6.9
             Utilities                      13.6          53.3            7.6
             Wholesale Trade                10.1          67.7            7.1
Size         Large (250+)                   15.8          27.6            6.3
             Medium (50-249)                16.3          40.1            5.8
             Micro (0-9)                     9.1          67.9            7.9
             Small (10-49)                  16.7          50.9            5.0

### Findings
- **Zero-growth norm**: Median DHS growth is effectively zero for most categories — stagnation is the modal incumbent outcome.
- **Declining HGF share**: The share of high-growth incumbents has trended down since the early 2000s, with a post-GFC step change downward.
- **Rising stagnation**: The stagnant share increased markedly post-GFC and again in 2020, suggesting the reallocation mechanism is weakening.
- **Size**: Micro firms have the highest mean growth and widest dispersion; large firms grow most slowly and with least variance.
- **Age**: Young firms (0–2 yrs) have the highest mean DHS growth and the highest HGF share — the age-growth gradient is steep.
- **Sector**: IT & Computer Services and Business Support Services have the highest HGF shares.
- **Productivity**: Frontier firms (P90+) have higher mean growth *and* higher dispersion; laggards have near-zero mean but also show shrinking.

### Remaining questions
- Is the decline in HGF share driven by fewer firms reaching high-growth, or by high-growth episodes becoming shorter?
- Which sectors show the largest rises in stagnant share post-2008 — is it concentrated in sectors with large incumbent firms?
- Does the productivity category show divergence — frontier growing faster while laggards stagnate?


---
## 4. Job Reallocation Rates

All rates use lagged (t−1) employment as the denominator.

- **JRR** (job reallocation rate) = (JC_entrants + JC_incumbents + JD_exiters + JD_incumbents) / emp_{t−1} × 100
- **Extensive margin** = (JC_entrants + JD_exiters) / emp_{t−1} × 100
- **Intensive margin** = (JC_incumbents + JD_incumbents) / emp_{t−1} × 100
- **JCR** = (JC_entrants + JC_incumbents) / emp_{t−1} × 100
- **JDR** = (JD_exiters + JD_incumbents) / emp_{t−1} × 100


In [145]:
jf2 = jf.sort_values(['dimension', 'category', 'year']).copy()
jf2['employment_lag'] = jf2.groupby(['dimension', 'category'])['employment'].shift(1)
# Extensive margin: entry + exit contribution combined
jf2['jr_rate_entry_exit']  = (jf2['job_creation_entrants']    + jf2['job_destruction_exiters'])   / jf2['employment_lag'] * 100
# Intensive margin: incumbent contribution combined
jf2['jr_rate_incumbents']  = (jf2['job_creation_incumbents']  + jf2['job_destruction_incumbents'])/ jf2['employment_lag'] * 100
# Total reallocation and creation/destruction rates
jf2['job_reallocation_rate'] = jf2['jr_rate_entry_exit'] + jf2['jr_rate_incumbents']
jf2['job_creation_rate']   = (jf2['job_creation_entrants']    + jf2['job_creation_incumbents'])   / jf2['employment_lag'] * 100
jf2['job_destruction_rate']= (jf2['job_destruction_exiters']  + jf2['job_destruction_incumbents'])/ jf2['employment_lag'] * 100
jf2['net_jrr']             = jf2['job_creation_rate'] - jf2['job_destruction_rate']
# Component rates
jf2['jc_rate_entrants']    = jf2['job_creation_entrants']    / jf2['employment_lag'] * 100
jf2['jc_rate_incumbents']  = jf2['job_creation_incumbents']  / jf2['employment_lag'] * 100
jf2['jd_rate_exiters']     = jf2['job_destruction_exiters']  / jf2['employment_lag'] * 100
jf2['jd_rate_incumbents']  = jf2['job_destruction_incumbents']/ jf2['employment_lag'] * 100
jf2 = jf2.dropna(subset=['employment_lag'])


In [146]:
# Total: extensive vs intensive margins
tot_jf  = jf2[jf2['dimension'] == 'Total'].copy()
jf_long = tot_jf.melt(id_vars=['year'],
    value_vars=['jr_rate_entry_exit', 'jr_rate_incumbents'],
    var_name='margin', value_name='rate')
jf_long['margin'] = jf_long['margin'].map({
    'jr_rate_entry_exit': 'Entry & exit (extensive)',
    'jr_rate_incumbents': 'Incumbents (intensive)'})

alt.Chart(jf_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('rate:Q', title='% of t−1 employment'),
    color=alt.Color('margin:N',
        scale=alt.Scale(domain=['Entry & exit (extensive)', 'Incumbents (intensive)'],
                        range=[pallete['nominal_1'], pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O', 'margin:N', alt.Tooltip('rate:Q', format='.1f')]
).properties(title='Job reallocation: extensive vs intensive margins, 2000–2022',
             width=680, height=320)


alt.Chart(...)

In [147]:
# JCR, JDR, net job growth
jrr_long = tot_jf.melt(id_vars=['year'],
    value_vars=['job_creation_rate', 'job_destruction_rate', 'net_jrr'],
    var_name='measure', value_name='rate')
jrr_long['measure'] = jrr_long['measure'].map(
    {'job_creation_rate': 'Job creation rate',
     'job_destruction_rate': 'Job destruction rate',
     'net_jrr': 'Net job growth'})

alt.Chart(jrr_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('rate:Q', title='% of t−1 employment', scale=alt.Scale(zero=False)),
    color=alt.Color('measure:N',
        scale=alt.Scale(domain=['Job creation rate', 'Job destruction rate', 'Net job growth'],
                        range=[pallete['nominal_1'], pallete['nominal_2'], pallete['nominal_3']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O', 'measure:N', alt.Tooltip('rate:Q', format='.1f')]
).properties(title='Job creation, destruction and net growth, 2000–2022', width=620, height=300)


alt.Chart(...)

In [148]:
# Period averages
period_means(tot_jf, ['job_creation_rate', 'job_destruction_rate', 'job_reallocation_rate',
                       'net_jrr', 'jr_rate_entry_exit', 'jr_rate_incumbents'])\
    .rename(columns={'job_creation_rate': 'JCR (%)', 'job_destruction_rate': 'JDR (%)',
                     'job_reallocation_rate': 'JRR (%)', 'net_jrr': 'Net (pp)',
                     'jr_rate_entry_exit': 'Extensive (%)', 'jr_rate_incumbents': 'Intensive (%)'})


,JCR (%),JDR (%),JRR (%),Net (pp),Extensive (%),Intensive (%)
Period,,,,,,
Pre-GFC (1999-2007),12.66,12.09,24.75,0.57,10.33,14.41
GFC (2008-2010),10.94,11.07,22.01,-0.12,8.09,13.92
Recovery (2011-2019),10.71,9.16,19.87,1.55,7.51,12.36
COVID & post (2020-2022),8.87,9.16,18.03,-0.29,6.62,11.40


In [149]:
# JRR by category — interactive
dim_sel4 = alt.binding_select(options=DIMS, name='Dimension: ')
dim_par4 = alt.selection_point(fields=['dimension'], bind=dim_sel4, value='Size')

cat_jf = jf2[jf2['dimension'].isin(DIMS)].copy()

jrr_c = alt.Chart(cat_jf).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('job_reallocation_rate:Q', title='JRR (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O', 'category:N', alt.Tooltip('job_reallocation_rate:Q', format='.1f')]
).add_params(dim_par4).transform_filter(dim_par4)\
 .properties(title='Job reallocation rate by category', width=620, height=280)

ext_c = alt.Chart(cat_jf).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('jr_rate_entry_exit:Q', title='Extensive margin (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O', 'category:N', alt.Tooltip('jr_rate_entry_exit:Q', format='.1f')]
).add_params(dim_par4).transform_filter(dim_par4)\
 .properties(title='Entry & exit margin (JC entrants + JD exiters)', width=620, height=280)

(jrr_c & ext_c)


alt.VConcatChart(...)

In [150]:
# Summary table
(jf2[jf2['dimension'].isin(DIMS)]
 .groupby(['dimension', 'category'])
 [['job_creation_rate', 'job_destruction_rate', 'job_reallocation_rate',
   'jr_rate_entry_exit', 'jr_rate_incumbents']]
 .mean().round(1)
 .rename(columns={'job_creation_rate': 'JCR (%)', 'job_destruction_rate': 'JDR (%)',
                  'job_reallocation_rate': 'JRR (%)',
                  'jr_rate_entry_exit': 'Extensive (%)', 'jr_rate_incumbents': 'Intensive (%)'}))


JCR (%)  JDR (%)  JRR (%)  \
dimension    category                                                
Age          Mature (6-10 years)             9.7     12.8     22.6   
             New (0-2 years)                40.2     15.9     56.1   
             Old (11+ years)                 6.4      8.4     14.8   
             Young (3-5 years)              12.8     16.8     29.6   
Productivity Frontier (P90+)                 7.5     14.7     22.2   
             High-Median (P50-P90)          10.2      9.0     19.1   
             Laggards (<P10)                16.9     11.3     28.2   
             Low-Median (P10-P50)           12.4     10.1     22.5   
Region       East Midlands                   9.8      9.8     19.6   
             East Of England                10.4      9.4     19.9   
             London                         12.4     10.8     23.2   
             North East                     11.2     11.0     22.2   
             North West                     11.7     10.8     22.5   
             Northern Ireland               10.2      9.5     19.7   
             Scotland                       10.9     10.5     21.4   
             South East                     11.4     10.8     22.2   
             South West                     11.4     10.7     22.0   
             Wales                          10.7     10.3     21.0   
             West Midlands                  10.9     10.8     21.6   
             Yorkshire and The Humber       10.4      9.8     20.1   
Sector       Agriculture                     8.6      9.7     18.3   
             Business Support Services      15.7     14.1     29.8   
             Construction                   13.9     12.5     26.4   
             Hospitality                    14.1     12.6     26.7   
             IT & Computer Services         17.2     14.0     31.2   
             Manufacturing                   7.5      8.9     16.4   
             Other Information Services     11.3     10.8     22.1   
             Other Primary Industries       10.3     12.7     23.0   
             Other Services                 12.2     10.8     23.0   
             Professional Services          14.0     12.0     25.9   
             Retail Trade                    8.0      7.2     15.2   
             Social care                    11.0      9.2     20.1   
             Transport & Logistics           8.3      7.8     16.1   
             Utilities                      12.4     10.2     22.5   
             Wholesale Trade                 9.2      9.3     18.5   
Size         Large (250+)                    9.1      7.4     16.5   
             Medium (50-249)                11.6     10.1     21.6   
             Micro (0-9)                    14.5     17.2     31.7   
             Small (10-49)                  12.0     10.0     22.0   

                                         Extensive (%)  Intensive (%)  
dimension    category                                                  
Age          Mature (6-10 years)                   6.7           15.9  
             New (0-2 years)                      39.5           16.7  
             Old (11+ years)                       3.3           11.5  
             Young (3-5 years)                     9.9           19.7  
Productivity Frontier (P90+)                       4.7           17.4  
             High-Median (P50-P90)                 8.0           11.1  
             Laggards (<P10)                      11.0           17.3  
             Low-Median (P10-P50)                 10.0           12.5  
Region       East Midlands                         7.4           12.2  
             East Of England                       7.3           12.5  
             London                                8.8           14.4  
             North East                            9.6           12.6  
             North West                            9.2           13.3  
             Northern Ireland                      7.0           12.7  
             Scotl

### Findings
- **Falling JRR**: Total job reallocation has fallen from ~55–60% in the late 1990s to ~35–40% by the 2010s — a substantial reduction in labour market fluidity.
- **Extensive margin collapse**: The entry/exit contribution to reallocation has fallen more than the intensive margin; smaller and fewer entrants are the main driver.
- **GFC**: Large JDR spike in 2008–09 from exiters; JCR via entry also fell sharply. Recovery was slow.
- **COVID**: JRR hit its record low in 2020 — both JCR and JDR fell as policy kept firms alive and hiring paused.
- **Size**: Micro firms have overwhelmingly the highest JRR (mostly extensive margin); large firms reallocation is almost entirely intensive.
- **Age**: New firms contribute a disproportionate share of gross job creation via the extensive margin.
- **Sector**: Hospitality and Retail show the highest JRR; Manufacturing and Utilities the lowest.
- **Productivity**: Laggard firms show surprisingly high JDR; frontier firms have the highest JCR.

### Remaining questions
- Is the falling JRR driven by compositional shifts (fewer micro/young firms) or within-category within-period declines?
- What is driving the extensive margin fall specifically — lower entry rates, smaller entrant size, or both?
- Has the gap between JCR and JDR at different points in the productivity distribution widened, suggesting misallocation?


---
## 5. Role of Young Firms

Young firms (0–5 years) are hypothesised to be disproportionately important for dynamism. This section assesses their share of firms, employment, and job creation, and how these shares have changed.


In [151]:
age_pop   = pop[pop['dimension'] == 'Age'].copy()
total_pop = (pop[pop['dimension'] == 'Total'][['year','n_firms','employment']]
             .rename(columns={'n_firms':'total_firms','employment':'total_emp'}))
age_pop = age_pop.merge(total_pop, on='year')
age_pop['firm_share'] = age_pop['n_firms']    / age_pop['total_firms'] * 100
age_pop['emp_share']  = age_pop['employment'] / age_pop['total_emp']   * 100

AGE_ORDER  = ['New (0-2 years)','Young (3-5 years)','Mature (6-10 years)','Old (11+ years)']
AGE_COLORS = [pallete['nominal_1'],pallete['nominal_3'],pallete['nominal_4'],pallete['nominal_2']]


In [152]:
# Stacked area: firm share by age group
alt.Chart(age_pop).mark_area().encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('firm_share:Q', title='Share of firms (%)', stack='normalize'),
    color=alt.Color('category:N', sort=AGE_ORDER,
        scale=alt.Scale(domain=AGE_ORDER, range=AGE_COLORS),
        legend=alt.Legend(title='')),
    order=alt.Order('category:N', sort='ascending'),
    tooltip=['year:O','category:N',alt.Tooltip('firm_share:Q',format='.1f')]
).properties(title='Age composition of the firm population (share of firms)', width=620, height=300)


alt.Chart(...)

In [153]:
# Stacked area: employment share by age group
alt.Chart(age_pop).mark_area().encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('emp_share:Q', title='Share of employment (%)', stack='normalize'),
    color=alt.Color('category:N', sort=AGE_ORDER,
        scale=alt.Scale(domain=AGE_ORDER, range=AGE_COLORS),
        legend=alt.Legend(title='')),
    order=alt.Order('category:N', sort='ascending'),
    tooltip=['year:O','category:N',alt.Tooltip('emp_share:Q',format='.1f')]
).properties(title='Age composition of the firm population (share of employment)', width=620, height=300)


alt.Chart(...)

In [154]:
# Young firm (0-5) combined share trend
young = (age_pop[age_pop['category'].isin(['New (0-2 years)','Young (3-5 years)'])]
         .groupby('year')[['n_firms','employment','total_firms','total_emp']].sum().reset_index())
young['firm_share'] = young['n_firms']    / young['total_firms'] * 100
young['emp_share']  = young['employment'] / young['total_emp']   * 100

y_long = young.melt(id_vars=['year'], value_vars=['firm_share','emp_share'],
                    var_name='measure', value_name='share')
y_long['measure'] = y_long['measure'].map(
    {'firm_share':'Share of firms','emp_share':'Share of employment'})

alt.Chart(y_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('share:Q', title='Share (%)', scale=alt.Scale(zero=False)),
    color=alt.Color('measure:N',
        scale=alt.Scale(domain=['Share of firms','Share of employment'],
                        range=[pallete['nominal_1'],pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O','measure:N',alt.Tooltip('share:Q',format='.1f')]
).properties(title='Young firms (0–5 years): share of firms and employment', width=620, height=300)


alt.Chart(...)

In [155]:
# Share of job creation by age group
age_jf  = jf2[jf2['dimension'] == 'Age'].copy()
tot_jc  = (jf2[jf2['dimension'] == 'Total'][['year','job_creation_entrants','job_creation_incumbents']]
           .assign(total_jc=lambda d: d['job_creation_entrants']+d['job_creation_incumbents']))
age_jf  = age_jf.merge(tot_jc[['year','total_jc']], on='year')
age_jf['jc_share'] = (age_jf['job_creation_entrants'] + age_jf['job_creation_incumbents']) / age_jf['total_jc'] * 100

alt.Chart(age_jf).mark_area().encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('jc_share:Q', title='Share of total job creation (%)', stack='normalize'),
    color=alt.Color('category:N', sort=AGE_ORDER,
        scale=alt.Scale(domain=AGE_ORDER, range=AGE_COLORS),
        legend=alt.Legend(title='')),
    order=alt.Order('category:N', sort='ascending'),
    tooltip=['year:O','category:N',alt.Tooltip('jc_share:Q',format='.1f')]
).properties(title='Share of gross job creation by firm age group', width=620, height=300)


alt.Chart(...)

In [156]:
# Growth rates by age group
age_gr = gr[gr['dimension'] == 'Age'].copy()

alt.Chart(age_gr).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('mean_dhs_growth:Q', title='Mean DHS growth rate'),
    color=alt.Color('category:N', sort=AGE_ORDER,
        scale=alt.Scale(domain=AGE_ORDER, range=AGE_COLORS),
        legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('mean_dhs_growth:Q',format='.3f')]
).properties(title='Mean DHS growth rate by firm age group', width=620, height=300)


alt.Chart(...)

In [157]:
# Period summary: young firm shares
rows = []
for label, (y0, y1) in PERIODS.items():
    sub = young[(young['year'] >= y0) & (young['year'] <= y1)]
    rows.append({'Period': label,
                 'Firm share (%)': sub['firm_share'].mean(),
                 'Emp share (%)':  sub['emp_share'].mean()})
pd.DataFrame(rows).set_index('Period').round(1)


,Firm share (%),Emp share (%)
Period,,
Pre-GFC (1999-2007),24.4,11.7
GFC (2008-2010),24.2,9.8
Recovery (2011-2019),23.4,9.3
COVID & post (2020-2022),23.4,9.5


### Findings
- **Ageing firm population**: The share of young firms (0–5 yrs) in total firm counts has fallen steadily — the firm population is getting older.
- **Employment underrepresentation**: Young firms account for a smaller share of employment than of firm numbers, reflecting their smaller average size.
- **Disproportionate gross job creation**: Despite small employment shares, new entrants account for a large share of gross job creation via the extensive margin.
- **Young firm growth premium**: Mean DHS growth is consistently highest for the 0–2 year cohort; this premium has narrowed post-GFC.
- **Post-GFC fall**: Young firm shares fell sharply after 2008, consistent with prolonged suppression of entry rates.

### Remaining questions
- Is the decline in young firm employment share driven by fewer young firms or by entrants being born smaller?
- Which sectors drive the ageing of the firm population — where have young firm shares fallen most?
- How does the UK young firm employment share compare with international benchmarks?


---
## 6. Productivity Dispersion

Measured as **turnover per employee** (£ nominal). Dispersion is captured by percentile ratios (P90/P10, P75/P25). Data: `prod` sheet (covers size, age, sector, region; **no productivity-ranking breakdown** since that dimension defines the variable).


In [158]:
prod2 = prod.copy()
prod2['p90_p10'] = prod2['p90_prod'] / prod2['p10_prod']
prod2['p75_p25'] = prod2['p75_prod'] / prod2['p25_prod']

tot_prod = prod2[prod2['dimension'] == 'Total'].copy()


In [159]:
# P90/P10 and P75/P25 ratios over time
disp_long = tot_prod.melt(id_vars=['year'], value_vars=['p90_p10','p75_p25'],
                           var_name='measure', value_name='ratio')
disp_long['measure'] = disp_long['measure'].map({'p90_p10':'P90/P10','p75_p25':'P75/P25'})

alt.Chart(disp_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('ratio:Q', title='Ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('measure:N',
        scale=alt.Scale(domain=['P90/P10','P75/P25'],
                        range=[pallete['nominal_1'],pallete['nominal_2']]),
        legend=alt.Legend(title='')),
    tooltip=['year:O','measure:N',alt.Tooltip('ratio:Q',format='.1f')]
).properties(title='Productivity dispersion: P90/P10 and P75/P25 ratios (total economy)',
             width=620, height=300)


alt.Chart(...)

In [160]:
# Percentile levels over time
pct_long = tot_prod.melt(id_vars=['year'],
    value_vars=['p10_prod','p25_prod','p50_prod','p75_prod','p90_prod'],
    var_name='pct', value_name='value')
pct_long['pct'] = pct_long['pct'].map(
    {'p10_prod':'P10','p25_prod':'P25','p50_prod':'P50','p75_prod':'P75','p90_prod':'P90'})

alt.Chart(pct_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('value:Q', title='Turnover/employee (£)', scale=alt.Scale(zero=False)),
    color=alt.Color('pct:N', legend=alt.Legend(title='Percentile')),
    tooltip=['year:O','pct:N',alt.Tooltip('value:Q',format=',.0f')]
).properties(title='Productivity distribution: absolute levels by percentile', width=620, height=300)


alt.Chart(...)

In [161]:
# By category — interactive
PROD_DIMS = ['Size','Age','Sector','Region']
dim_sel5 = alt.binding_select(options=PROD_DIMS, name='Dimension: ')
dim_par5 = alt.selection_point(fields=['dimension'], bind=dim_sel5, value='Sector')

cat_prod = prod2[prod2['dimension'].isin(PROD_DIMS)].copy()

disp_c = alt.Chart(cat_prod).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('p90_p10:Q', title='P90/P10 ratio', scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('p90_p10:Q',format='.1f')]
).add_params(dim_par5).transform_filter(dim_par5) .properties(title='Productivity dispersion (P90/P10) by category', width=620, height=300)

disp_c


alt.Chart(...)

In [162]:
# Cross-section bar: latest year
latest = prod2['year'].max()
prod_lat = prod2[(prod2['year']==latest) & prod2['dimension'].isin(PROD_DIMS)].copy()

alt.Chart(prod_lat).mark_bar().encode(
    x=alt.X('p90_p10:Q', title='P90/P10 ratio'),
    y=alt.Y('category:N', sort='-x', title=''),
    color=alt.Color('dimension:N', legend=alt.Legend(title='Dimension')),
    tooltip=['dimension:N','category:N',alt.Tooltip('p90_p10:Q',format='.1f')]
).properties(title=f'Productivity dispersion by category ({latest})', width=500, height=420)


alt.Chart(...)

In [163]:
# Productivity by continuous firm age (averaged across years)
age_c = prod2[(prod2['dimension']=='Age continuous') & (prod2['category']!='10+')].copy()
age_c['category'] = pd.to_numeric(age_c['category'], errors='coerce')
age_c = age_c.dropna(subset=['category'])
age_avg = age_c.groupby('category')[['p10_prod','p50_prod','p90_prod']].mean().reset_index()

p50_l = alt.Chart(age_avg).mark_line(color=pallete['nominal_1'], strokeWidth=2, point=True).encode(
    x=alt.X('category:Q', title='Firm age (years)'),
    y=alt.Y('p50_prod:Q', title='Turnover/employee (£)'),
    tooltip=['category:Q',alt.Tooltip('p50_prod:Q',format=',.0f')])

band3 = alt.Chart(age_avg).mark_area(opacity=0.15, color=pallete['nominal_1']).encode(
    x='category:Q', y='p10_prod:Q', y2='p90_prod:Q')

(band3 + p50_l).properties(
    title='Productivity by continuous firm age (avg 1997–2023; band = P10–P90)',
    width=500, height=300)


alt.LayerChart(...)

In [164]:
# Summary table
(prod2[prod2['dimension'].isin(PROD_DIMS)]
 .groupby(['dimension','category'])[['p90_p10','p75_p25','p50_prod']]
 .mean().round(1)
 .rename(columns={'p90_p10':'Avg P90/P10','p75_p25':'Avg P75/P25',
                  'p50_prod':'Avg median prod (£)'}))


Avg P90/P10  Avg P75/P25  \
dimension category                                               
Age       Mature (6-10 years)                10.1          3.0   
          New (0-2 years)                     inf          2.5   
          Old (11+ years)                    12.1          3.2   
          Young (3-5 years)                   9.4          2.9   
Region    East Midlands                       9.2          2.9   
          East Of England                     9.2          2.8   
          London                              9.9          2.8   
          North East                          7.7          2.8   
          North West                          8.3          2.8   
          Northern Ireland                   17.7          3.9   
          Scotland                            8.7          2.9   
          South East                          9.2          2.9   
          South West                          9.3          2.9   
          Wales                              10.5          3.1   
          West Midlands                       9.0          2.9   
          Yorkshire and The Humber            8.6          2.9   
Sector    Agriculture                         inf          4.8   
          Business Support Services           9.5          3.1   
          Construction                       10.9          2.8   
          Hospitality                         4.0          1.7   
          IT & Computer Services              6.3          2.0   
          Manufacturing                       7.4          2.4   
          Other Information Services         13.1          3.0   
          Other Primary Industries           23.9          4.1   
          Other Services                      7.7          2.9   
          Professional Services               6.8          2.4   
          Retail Trade                        7.0          2.3   
          Social care                         2.0          1.3   
          Transport & Logistics               7.7          2.6   
          Utilities                          14.0          2.9   
          Wholesale Trade                    15.1          4.0   
Size      Large (250+)                       26.2          5.0   
          Medium (50-249)                    16.9          3.6   
          Micro (0-9)                         9.0          2.9   
          Small (10-49)                      12.1          3.2   

                                      Avg median prod (£)  
dimension category                                         
Age       Mature (6-10 years)                        59.7  
          New (0-2 years)                            63.4  
          Old (11+ years)                            55.4  
          Young (3-5 years)                          59.9  
Region    East Midlands                              57.4  
          East Of England                            61.3  
          London                                     70.9  
          North East                                 54.5  
          North West                                 58.5  
          Northern Ireland                           46.8  
          Scotland                                   55.0  
          South East                                 61.7  
          South West                                 54.6  
          Wales                                      49.9  
          West Midlands                              58.6  
          Yorkshire and The Humber                   56.8  
Sector    Agriculture                                45.7  
          Business Support Services                  74.5  
          Construction                               75.3  
          Hospitality                                33.6  
          IT & Computer Services                     63.4  
          Manufacturing                              60.7  
          Other Information Services                 68.8  
          Other Primary Industries                   56.6  
          Other Services           

### Findings
- **High and persistent dispersion**: The P90/P10 ratio is ~12–15× and has shown no sustained narrowing — firm-level productivity heterogeneity is large and stable.
- **Widening top tail**: P90 has grown faster than P50 in nominal terms; P10 has barely moved — consistent with a frontier-divergence pattern.
- **Sector**: IT & Computer Services and Professional Services show the widest within-sector dispersion; Agriculture and Social Care the narrowest.
- **Size**: Large firms have higher median productivity, but micro firms have the widest P90/P10 within their size class.
- **Age**: Median productivity rises with age up to around 6–8 years then plateaus; the P10–P90 band also widens with age, indicating greater within-age-group heterogeneity among older firms.
- **Region**: London has the highest median productivity and the highest within-region dispersion.

### Remaining questions
- Is rising P90/P10 driven by the top growing faster, the bottom stagnating, or both simultaneously?
- Does the productivity–age profile differ by sector — do knowledge-intensive firms show faster early-life productivity growth?
- How does within-sector dispersion compare with between-sector dispersion in explaining total economy-wide dispersion?
- Is the productivity gap between frontier and laggard firms correlated with the dynamism gaps identified in sections 1–5?


---
## 7. Cross-cutting: Dynamism by Productivity Ranking

Are high- and low-productivity firms dynamically different, and has this relationship changed?


In [165]:
prod_fd  = fd2[fd2['dimension']=='Productivity'].copy()
prod_jf2 = jf2[jf2['dimension']=='Productivity'].copy()
prod_gr2 = gr[gr['dimension']=='Productivity'].copy()
prod_gc2 = gc2[gc2['dimension']=='Productivity'].copy()

# Entry and exit by productivity quartile
entry_p = alt.Chart(prod_fd).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('entry_rate:Q', title='Entry rate (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('entry_rate:Q',format='.1f')]
).properties(title='Entry rate by productivity quartile', width=620, height=270)

exit_p = alt.Chart(prod_fd).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('exit_rate:Q', title='Exit rate (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O','category:N',alt.Tooltip('exit_rate:Q',format='.1f')]
).properties(title='Exit rate by productivity quartile', width=620, height=270)

(entry_p & exit_p)


alt.VConcatChart(...)

In [166]:
# JRR and HGF share by productivity quartile
jrr_p = alt.Chart(prod_jf2).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('job_reallocation_rate:Q', title='JRR (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O', 'category:N', alt.Tooltip('job_reallocation_rate:Q', format='.1f')]
).properties(title='Job reallocation rate by productivity quartile', width=620, height=270)

hgf_p = alt.Chart(prod_gc2).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('year:O', title=''),
    y=alt.Y('hgf_sh:Q', title='HGF share (%)'),
    color=alt.Color('category:N', legend=alt.Legend(title='')),
    tooltip=['year:O', 'category:N', alt.Tooltip('hgf_sh:Q', format='.1f')]
).properties(title='HGF share by productivity quartile', width=620, height=270)

(jrr_p & hgf_p)


alt.VConcatChart(...)

In [167]:
# Summary: all measures by productivity quartile
prod_summary = pd.DataFrame({
    'Avg entry rate (%)':     prod_fd.groupby('category')['entry_rate'].mean(),
    'Avg exit rate (%)':      prod_fd.groupby('category')['exit_rate'].mean(),
    'Avg JRR (%)':            prod_jf2.groupby('category')['job_reallocation_rate'].mean(),
    'Avg mean DHS growth':    prod_gr2.groupby('category')['mean_dhs_growth'].mean(),
    'Avg HGF share (%)':      prod_gc2.groupby('category')['hgf_sh'].mean(),
    'Avg stagnant share (%)': prod_gc2.groupby('category')['stag_sh'].mean(),
}).round(2)

cat_order = ['Laggards (<P10)', 'Low-Median (P10-P50)', 'High-Median (P50-P90)', 'Frontier (P90+)']
prod_summary.loc[cat_order]


,Avg entry rate (%),Avg exit rate (%),Avg JRR (%),Avg mean DHS growth,Avg HGF share (%),Avg stagnant share (%)
category,,,,,,
Laggards (<P10),8.76,18.45,28.23,0.04,9.96,71.54
Low-Median (P10-P50),13.75,13.42,22.48,0.03,10.05,66.95
High-Median (P50-P90),15.72,12.77,19.14,0.01,10.55,62.52
Frontier (P90+),6.93,8.78,22.15,-0.05,8.78,64.11


### Findings
- **Non-linear dynamism–productivity relationship**: The highest entry rates belong to High-Median firms (P50–P90, avg 12.3%), not frontier firms. Frontier (P90+) entry rates are low (5.6%), suggesting few new entrants immediately reach the top of the productivity distribution — firms appear to start in the middle quartiles and either grow into the frontier or stagnate.
- **Laggard exit is the clearest selection signal**: Laggard firms (<P10) have by far the highest exit rate (15.1%), roughly double that of frontier firms (7.4%) — consistent with market selection operating, though the persistence of many laggards suggests it operates slowly.
- **Middle quartiles are the stagnation zone**: Low-Median and High-Median firms have the highest stagnant shares and moderate JRR — these are the incumbent firms that neither grow into the frontier nor exit.
- **JRR declines across all productivity groups over the period**, consistent with the aggregate decline in reallocation being broad-based rather than concentrated in one part of the distribution.

### Remaining questions
- Has the laggard exit rate slowed post-GFC (zombie accumulation), or is the 15% average roughly stable across periods?
- Why do frontier firms have such low entry rates — is this a measurement artefact of how the productivity quartiles are defined (incumbents only, ranked in each year), or a genuine structural feature?
- Do High-Median entrants systematically graduate to the frontier over time, or do most stagnate in the middle quartiles?
